# MÅL

Bruke reddits webapi til å:
- gjøre et søk på "elbiler"
- Hente alle artikler ("treff") fra det siste året
- Hente ut alle (nesten) kommentarer til innleggene
- putte det i pandas dataframe

evt:
- Gjøre sentimentanalyse med chatgpt api positiv/nøytral/negativ
- plotte utvikling (om noen)

In [6]:
import requests, json
import pandas as pd


In [201]:
with open("reddit_tokens.json", "r") as file:
    tokens = json.load(file)

access_token = tokens["access_token"]
refresh_token = tokens["refresh_token"]

client_id = "v2uZeXUHIszhF2K4hNOksQ"
client_secret = "2qamP2_KAEG7eNkXMwrJPhbb6jxzKw"

def refresh_tokens():
    global access_token, refresh_token
    refresh_url = "https://www.reddit.com/api/v1/access_token"
    payload = {"grant_type": "refresh_token", "refresh_token": refresh_token}
    headers = {"User-Agent": "python:undervisning_h25"}
    res = requests.post(refresh_url, auth=(client_id, client_secret), data=payload, headers=headers)
    res.raise_for_status()
    tokens = res.json()
    access_token = tokens["access_token"]
    refresh_token = tokens["refresh_token"]
    with open("reddit_tokens.json", "w") as file:
        json.dump(tokens,file)


def reddit_get(endpoint, params=None):
    base_url = "https://oauth.reddit.com"
    headers = {"Authorization": f"Bearer {access_token}",
              "User-Agent": "python:undervisning_h25"}
    res = requests.get(base_url+endpoint, headers=headers, params=params)
    res.raise_for_status()
    return res.json()

def reddit_search(q):
    params =  {
        "q": q,
        "sort": "top",
        "t": "year",
        "limit": 100
    }
    res_sider = []
    res = reddit_get("/search", params)
    res_sider.append(res)
    after = res["data"]["after"]
    
    while after:
        params = {
            "q": q,
            "sort": "top",
            "t": "year",
            "limit": 100,
            "after": after
        }
        res = reddit_get("/search", params)
        res_sider.append(res)
        after = res["data"]["after"]
    return res_sider



In [63]:
params = {
    "q": "elbil",
    "sort": "top",
    "t": "year",
    "limit": 100
}

treff = reddit_search("elbil")



In [87]:
dfs = [ pd.json_normalize(dat, record_path=["data","children"]) for dat in treff]
df = pd.concat(dfs)
kolonner = ["kind", "data.subreddit", "data.selftext", 
            "data.author_fullname", "data.title", 
            "data.name", "data.id", "data.created", "data.url"]
df_search = df[kolonner]
df_search = df_search.set_index(pd.PeriodIndex(pd.to_datetime(df_search["data.created"],unit="s", utc=True), freq="D"))
df_search = df_search.sort_index()
df_search

,kind,data.subreddit,data.selftext,data.author_fullname,data.title,data.name,data.id,data.created,data.url
data.created,,,,,,,,,
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...
2024-12-22,t3,Salamanders40k,Hello everyone. I Am brand new to Warhammer 40...,t2_8w0u6qkh7,New guy to Warhammer.,t3_1hk1oau,1hk1oau,1.734884e+09,https://i.redd.it/ogid7eakdf8e1.jpeg
2024-12-22,t3,elbilsverige,Helt sen jag investerade i en Elbil har jag gå...,t2_fea2scsox,"Elbil, en trevlig resa",t3_1hk19x3,1hk19x3,1.734883e+09,https://www.reddit.com/r/elbilsverige/comments...
2024-12-26,t3,elbilsverige,Ska strax påbörja andra resan till fjällen med...,t2_d7idzowy,Andra fjällresan,t3_1hmm9ed,1hmm9ed,1.735211e+09,https://www.reddit.com/r/elbilsverige/comments...
2024-12-27,t3,sweden,Finns alltid några procent extra kvar under hu...,t2_d7idzowy,Den bästa anledningen till att elbil är bättre...,t3_1hncxk3,1hncxk3,1.735299e+09,https://i.redd.it/6v9tadh5md9e1.jpeg
...,...,...,...,...,...,...,...,...,...
2025-12-11,t3,elbilsverige,,t2_5jwb7,EU ger grönt ljus till Sveriges sociala klimat...,t3_1pk7926,1pk7926,1.765483e+09,https://omni.se/eu-ger-gront-ljus-till-sverige...
2025-12-13,t3,dkbiler,"Jeg har ventet spændt på, at BMW får lanceret ...",t2_37utfpb2,Er der fremtid i elbiler i Danmark når den ful...,t3_1plgpyu,1plgpyu,1.765615e+09,https://www.reddit.com/r/dkbiler/comments/1plg...
2025-12-14,t3,HTML,,t2_1rwjy16cos,How would I make a website like Arngren.net?,t3_1pmmuoe,1pmmuoe,1.765741e+09,https://i.redd.it/h464paxl287g1.png


In [46]:
treff["data"]["children"][0]["data"]["selftext"]
#print("Antall treff", len(treff["data"]["children"]))
#for post in treff["data"]["children"]:
#    print(post["data"]["selftext"])
#    print("------------------------------\n\n")

'Etter å ha lest statistikken om hvor JÆVLIG mye Tesla vi nordmenn kjøper, kommer nok denne posten til å bli downvota til hælvete. Og jeg sier dette ekstra frustrert fordi foreldrene mine akkurat har kjøpt Tesla. Jeg vet de bare tenker på miljøet og økonomien. Men likevel, det føles bare så idiotisk. Salget i Europa har gått motsatt vei, men i Norge selges disse bilene som varmt hvetebrød... Vi i Norge liker å tro at vi er så moralsk overlegne når det kommer til amerikansk politikk. De aller fleste jeg snakker med synes DJT er en katastrofe. Egoisme, korrupsjon, ekkokamre, klimafornekting, splittelse. Likevel, når vi ser på hva som triller rundt på norske veier, er det Tesla overalt. Det er blitt nasjonalbilen? Og ja, jeg skjønner det er en elbil og at folk liker å tenke de gjør en miljøinnsats. Men på den andre siden, når du kjøper en Tesla, putter du penger rett i lomma på Elon Musk.\n\nOg Musk er grunnen til at DJT vant valget, og han flørter åpenlyst med høyreradikale miljøer, og s

In [180]:
kommentar[2]["data"]

IndexError: list index out of range

In [197]:

testid = df_search.iloc[0,-3]
url = df_search.iloc[0,-1]

comments_url = "/comments/article"
params = {"article": testid}

kommentar = reddit_get(comments_url, params)

with open("kommentartest.json", "w") as file:
    json.dump(kommentar,file)



HTTPError: 401 Client Error: Unauthorized for url: https://oauth.reddit.com/comments/article?article=1hj6tfs

In [187]:
kommentarer = kommentar[1]["data"]["children"].copy()
out = []
while len(kommentarer) > 0:
    kom = kommentarer.pop()
    if kom["kind"] == "Listing":
        kommentarer.extend(kom["data"]["children"])
    else:
        out.append(kom["data"]["body"])
        if isinstance(kom["data"]["replies"], dict):
            kommentarer.append(kom["data"]["replies"])



In [215]:
def get_comments(artikkel_id):
    comments_url = "/comments/article"
    params = {"article": artikkel_id}
    data = reddit_get(comments_url, params)
    kommentarer = data[1]["data"]["children"].copy()
    out = []
    while len(kommentarer) > 0:
        kom = kommentarer.pop()
        if kom["kind"] == "Listing":
            kommentarer.extend(kom["data"]["children"])
        elif kom["kind"] == "t1":
            out.append(kom["data"]["body"])
            if isinstance(kom["data"]["replies"], dict):
                kommentarer.append(kom["data"]["replies"])
    return out

testid = "1nqsvr8"
kommentarer = get_comments(testid)


In [221]:
df_search["kommentarer"] = df_search["data.id"].map(get_comments)


In [232]:
df_search.set_index("data.id")
n_comments = 243

n_comments += df_search["kommentarer"].map(len).sum()
print("antall innlegg + kommentarer = ", n_comments)

antall innlegg + kommentarer =  15446


In [241]:
df_search.index.name = "dato"
df = df_search.copy()

In [242]:
df = df.explode("kommentarer")
df

,kind,data.subreddit,data.selftext,data.author_fullname,data.title,data.name,data.id,data.created,data.url,kommentarer
dato,,,,,,,,,,
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...,De første mange afbetalinger på dit billån er ...
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...,Jeg vil bare hilse fra der skred fra Nordea i ...
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...,Jeg har den opfattelse er at den friværdi man ...
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...,Sælg bilen og køb en I har råd til istedet. Hv...
2024-12-21,t3,dkfinance,Vi har pt. 2 lån.\n\nLån 1:\nBoligpuls hos Nor...,t2_9tk8swgy,Låne i friværdi- betale billån ud,t3_1hj6tfs,1hj6tfs,1.734775e+09,https://www.reddit.com/r/dkfinance/comments/1h...,Hvem siger de ikke har råd til den? \n\nOP har...
...,...,...,...,...,...,...,...,...,...,...
2025-12-17,t3,PrivatEkonomi,"Jag och sambon har köpt en begagnad elbil, en ...",t2_316t3,Fel på beg elbil köpt för mindre än 6 månader ...,t3_1poxqbk,1poxqbk,1.765981e+09,https://i.redd.it/ys7pnx01xr7g1.jpeg,Som jag skrev finns ingen mejladress kopplat t...
2025-12-17,t3,PrivatEkonomi,"Jag och sambon har köpt en begagnad elbil, en ...",t2_316t3,Fel på beg elbil köpt för mindre än 6 månader ...,t3_1poxqbk,1poxqbk,1.765981e+09,https://i.redd.it/ys7pnx01xr7g1.jpeg,Rekbrev är också ett bra sätt.
2025-12-17,t3,PrivatEkonomi,"Jag och sambon har köpt en begagnad elbil, en ...",t2_316t3,Fel på beg elbil köpt för mindre än 6 månader ...,t3_1poxqbk,1poxqbk,1.765981e+09,https://i.redd.it/ys7pnx01xr7g1.jpeg,Ja SMS funkar också\n\nVatten i bakluckan är o...


In [247]:
%%time
a = 0
for i in range(1000):
    a += i
print("noe greier")

noe greier
CPU times: user 586 µs, sys: 0 ns, total: 586 µs
Wall time: 597 µs
